In [3]:
import mlflow
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri("https://tasha-interlinear-berneice.ngrok-free.dev")
mlflow.set_experiment(f"/user/test")

with mlflow.start_run() as run:

    mlflow.log_param("n_estimator", 1001)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("max_features", 3)

    # Load the diabetes dataset.
    db = load_diabetes()
    X_train, X_test, y_train, y_test = train_test_split(db.data, db.target)

    # Create and train models.
    rf = RandomForestRegressor(n_estimators=1001, max_depth=6, max_features=3)
    rf.fit(X_train, y_train)

    # Use the model to make predictions on the test dataset.
    predictions = rf.predict(X_test)

    mlflow.log_metric("accuracy", rf.score(X_test, y_test))

    mlflow.sklearn.log_model(rf, "model")

    print("RUN ID: {}".format(run.info.run_id) )

2026/09/15 17:51:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RUN ID: ab6bca3356724b1f904ed6e93c8de91a
🏃 View run crawling-sloth-987 at: http://localhost:5002/#/experiments/1/runs/ab6bca3356724b1f904ed6e93c8de91a
🧪 View experiment at: http://localhost:5002/#/experiments/1


In [20]:
import os
import mlflow
import mlflow.sklearn
import pandas as pd

# 1. Configurar credenciales hacia MinIO local
os.environ["AWS_ACCESS_KEY_ID"] = "minio_user"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minio_pwd"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:9000"
os.environ["MLFLOW_S3_IGNORE_TLS"] = "true"

# 2. Conectar al tracking server
mlflow.set_tracking_uri("http://localhost:5002")

# 3. Obtener dinámicamente el artifact_uri del run
run_id = "ab6bca3356724b1f904ed6e93c8de91a"
client = mlflow.tracking.MlflowClient()
run_info = client.get_run(run_id)

# Esto te da directamente la ruta real en s3://
artifact_uri = f"{run_info.info.artifact_uri}/model"

# 4. Cargar modelo y predecir
loaded_model = mlflow.sklearn.load_model(artifact_uri)
print("¡Modelo listo para inferencia!")

MlflowException: The following failures occurred while downloading one or more artifacts from http://localhost:5002/api/2.0/mlflow-artifacts/artifacts/1/ab6bca3356724b1f904ed6e93c8de91a/artifacts:
##### File model #####
404 Client Error: Not Found for url: http://localhost:5002/api/2.0/mlflow-artifacts/presigned/1/ab6bca3356724b1f904ed6e93c8de91a/artifacts/model. Response text: {"error_code": "RESOURCE_DOES_NOT_EXIST", "message": "Not Found", "sqlstate": "KAM00", "error_class": "RESOURCE_NOT_FOUND"}